# Layer-21 refit diagnostic on Gemma 4 E4B

**Question.** The frozen pilot lens's `J_21` is a massive scale outlier
(`||J_21||_F = 188.7` vs 3.5–18.1 at every other fitted layer; see
`docs/jspace_run_report.md` §4), and J-space explained fractions collapse
there. Is that (a) an unstable corpus-averaged fit, (b) dominance by a few
heavy-tailed prompts, or (c) a reproducible property of the layer?

**Design.** Repeat the pilot fit **exactly** — same model revision, the same
100 streamed WikiText-103 prompts in the same order (verified hash-by-hash
against the pilot's recorded `prompt_hashes`), seed 42, `max_seq_len` 128,
`dim_batch` 8, checkpoint every 5 prompts — but at **source layer 21 only**
(target layer 41), recording per-prompt Jacobian diagnostics
(`jlens/fit_diagnostics.py`): Frobenius norm, max |entry|, finiteness,
running mean/variance, per-prompt contribution and alignment. Afterwards run
only (1) the held-out v2 evaluation at layer 21 and (2) J-space gradient
pursuit at layer 21, **k = 10 only**, using the lower-memory helpers
(`validate_finite`, `build_chunk_rows`) so an L4 suffices.

**What this notebook never does:** refit or modify the pilot/smoke/jspace
runs (all reference reads are fingerprint-checked and read-only), fit any
other layer, run k = 16/25, use a chat-augmented corpus, or touch multimodal
inputs. Config: `configs/gemma_layer21_diagnostic.yaml`.

**Interpretation of outcomes** (descriptive, decided after the run):
a reproduced `||J_21||_F ≈ 189` with stable accumulation favours a real
property of the layer; a wildly different norm or a running mean that never
stabilizes favours estimator instability; a top-1/top-5 norm share close to
the total favours prompt dominance. These are diagnostics of *this
estimator on this corpus*, not model-internal claims by themselves.

## 0. Colab setup (skip if running locally)

In [ ]:
# 0. Colab bootstrap: clone/update the private repo from a fresh runtime.
# No-op outside Colab. Never loads Gemma; only touches git/pip.
import base64
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
    print("Not running in Colab — skipping bootstrap; using the local checkout.")
else:
    CHECKOUT_DIR = Path("/content/jacobian-lens-gemma")
    REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
    BRANCH = "layer21-diagnostic"

    def _normalize(url: str) -> str:
        return url.strip().removesuffix(".git").removesuffix("/")

    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception:
        GITHUB_TOKEN = None
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN secret not found or not accessible. In Colab: open "
            "the key icon (Secrets) in the left sidebar, add a secret named "
            "GITHUB_TOKEN containing a fine-grained GitHub token with "
            "read-only access to MechInterpreter/jacobian-lens-gemma, and "
            "enable notebook access for it. Then re-run this cell."
        )

    # Auth via a per-invocation extraHeader override: lives only in argv,
    # never written to .git/config, the remote URL, or notebook output.
    _token_b64 = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    _auth_header = f"AUTHORIZATION: basic {_token_b64}"
    _auth_args = ["-c", f"http.https://github.com/.extraHeader={_auth_header}"]

    def _run(args, *, auth=False, check=True):
        cmd = ["git", *(_auth_args if auth else []), *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if check and result.returncode != 0:
            safe_stderr = result.stderr.replace(GITHUB_TOKEN, "***")
            raise RuntimeError(f"git {' '.join(args)} failed:\n{safe_stderr}")
        return result.stdout.strip()

    if not CHECKOUT_DIR.exists():
        print(f"Cloning {REPO_URL} ({BRANCH}) into {CHECKOUT_DIR} ...")
        _run(["clone", "--branch", BRANCH, REPO_URL, str(CHECKOUT_DIR)], auth=True)
    else:
        if not (CHECKOUT_DIR / ".git").exists():
            raise RuntimeError(
                f"{CHECKOUT_DIR} exists but is not a git checkout; refusing to "
                "touch it. Remove or rename it manually, then re-run this cell."
            )
        existing_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
        if _normalize(existing_url) != _normalize(REPO_URL):
            raise RuntimeError(
                f"{CHECKOUT_DIR} is checked out from {existing_url!r}, not "
                f"{REPO_URL!r}; refusing to touch an unexpected repository. "
                "Remove or rename it manually, then re-run this cell."
            )
        print(f"Updating existing checkout at {CHECKOUT_DIR} ...")
        _run(["-C", str(CHECKOUT_DIR), "fetch", "origin"], auth=True)
        _run(["-C", str(CHECKOUT_DIR), "checkout", BRANCH])
        _run(["-C", str(CHECKOUT_DIR), "merge", "--ff-only", f"origin/{BRANCH}"])

    final_url = _run(["-C", str(CHECKOUT_DIR), "remote", "get-url", "origin"])
    if "@" in final_url or GITHUB_TOKEN in final_url:
        raise RuntimeError("origin URL unexpectedly contains credentials; aborting.")

    os.chdir(CHECKOUT_DIR)
    if str(CHECKOUT_DIR) not in sys.path:
        sys.path.insert(0, str(CHECKOUT_DIR))

    print("Installing the 'gemma' extra ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[gemma]"],
        check=True,
    )

    branch_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "--abbrev-ref", "HEAD"])
    sha_now = _run(["-C", str(CHECKOUT_DIR), "rev-parse", "HEAD"])
    print(f"checked out: {branch_now} @ {sha_now}")
    assert (CHECKOUT_DIR / "jlens").is_dir(), f"{CHECKOUT_DIR}/jlens not found"

In [ ]:
# 1. Execution gates. Model loading and CUDA are opt-in; outside Colab the
# defaults keep everything in the light (no-download) path so the notebook's
# structure can be validated locally.
import logging
import os
import sys

IN_COLAB = "google.colab" in sys.modules
os.environ.setdefault("JLENS_ALLOW_GEMMA", "1" if IN_COLAB else "0")
os.environ.setdefault("JLENS_DEVICE_MAP", "cuda" if IN_COLAB else "")
# Resuming a specific prior run is an explicit opt-in (its exact RUN_DIR):
# os.environ["L21DIAG_RESUME_RUN_DIR"] = ".../runs/layer21diag_..."
ALLOW_MODEL_LOAD = os.environ.get("JLENS_ALLOW_GEMMA", "0") == "1"
DEVICE_MAP = os.environ.get("JLENS_DEVICE_MAP") or None
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s %(message)s")
logging.getLogger("jlens").setLevel(logging.INFO)
print(f"IN_COLAB={IN_COLAB}  ALLOW_MODEL_LOAD={ALLOW_MODEL_LOAD}  DEVICE_MAP={DEVICE_MAP}")

In [ ]:
# 2. Environment and provenance (no model load).
import json
import pathlib
import time

import torch

from jlens.metadata import environment_manifest

ENV = environment_manifest()
print(json.dumps(ENV, indent=2))

In [ ]:
# 3. Google Drive persistence (Colab only). No-op outside Colab.
from pathlib import Path

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:
        raise RuntimeError(
            f"failed to mount Google Drive at /content/drive: {exc}. Approve "
            "the Drive authorization prompt when it appears, then re-run this cell."
        ) from exc

    DRIVE_MOUNT = Path("/content/drive")
    if not DRIVE_MOUNT.is_dir():
        raise RuntimeError("Drive did not mount successfully.")

    PERSIST_ROOT = DRIVE_MOUNT / "MyDrive" / "jacobian-lens-gemma"
    RUNS_ROOT = PERSIST_ROOT / "runs"
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"RUNS_ROOT = {RUNS_ROOT}")
else:
    PERSIST_ROOT = None
    RUNS_ROOT = Path("runs")  # local checkout: read the archived pilot run
    print("Not in Colab — using the local runs/ directory (read) and "
          "artifacts/layer21_diagnostic (write).")

In [ ]:
# 4. Load and validate the diagnostic configuration; hard-stop on any layer
# or scope drift; create a fresh timestamped run directory (or resume one,
# explicitly and fingerprint-validated, refusing completed runs).
from datetime import datetime, timezone

from jlens.metadata import config_fingerprint, load_config

CONFIG_PATH = "configs/gemma_layer21_diagnostic.yaml"
CONFIG = load_config(CONFIG_PATH)
FINGERPRINT = config_fingerprint(CONFIG)
print(f"config: {CONFIG_PATH}\nfingerprint: {FINGERPRINT}")

# STOP CONDITIONS — scope is layer 21 / k=10 only, by construction:
if CONFIG["mode"] != "layer21_diagnostic":
    raise RuntimeError(f"unexpected mode {CONFIG['mode']!r}")
if CONFIG["sites"]["source_layers"] != [21]:
    raise RuntimeError(
        f"source_layers must be [21] for this diagnostic, got "
        f"{CONFIG['sites']['source_layers']}"
    )
if CONFIG["sites"]["target_layer"] != 41:
    raise RuntimeError(f"target_layer must be 41, got {CONFIG['sites']['target_layer']}")
if CONFIG["pursuit"]["k_values"] != [10]:
    raise RuntimeError(f"pursuit k_values must be [10], got {CONFIG['pursuit']['k_values']}")
if CONFIG["model"]["revision"] != CONFIG["reference"]["expect_model_revision"]:
    raise RuntimeError("config model.revision != reference.expect_model_revision")

RESUMED = False
RESUME_RUN_DIR_ENV = os.environ.get("L21DIAG_RESUME_RUN_DIR") or None
if IN_COLAB:
    if RESUME_RUN_DIR_ENV is not None:
        RUN_DIR = pathlib.Path(RESUME_RUN_DIR_ENV)
        started_path = RUN_DIR / "run_started.json"
        if not started_path.is_file():
            raise RuntimeError(
                f"L21DIAG_RESUME_RUN_DIR={RUN_DIR} has no run_started.json; "
                "point it at a previously started layer21diag run directory"
            )
        started = json.load(open(started_path, encoding="utf-8"))
        if started.get("config_fingerprint") != FINGERPRINT:
            raise RuntimeError(
                f"refusing to resume {RUN_DIR}: recorded fingerprint "
                f"{started.get('config_fingerprint')!r} != active {FINGERPRINT!r}"
            )
        if (RUN_DIR / "run_metadata.json").is_file():
            raise RuntimeError(
                f"{RUN_DIR} already contains run_metadata.json — that run is "
                "COMPLETE. Refusing to overwrite it; start a fresh run instead."
            )
        RUN_ID = RUN_DIR.name
        RESUMED = True
        print(f"resuming run: {RUN_DIR} (fingerprint verified, not complete)")
    else:
        RUN_ID = (f"layer21diag_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')}_"
                  f"{FINGERPRINT.removeprefix('sha256:')[:12]}")
        RUN_DIR = RUNS_ROOT / RUN_ID
        RUN_DIR.mkdir(parents=True, exist_ok=False)  # fresh dir; never overwrite
        with open(RUN_DIR / "run_started.json", "w", encoding="utf-8") as fh:
            json.dump({
                "run_id": RUN_ID,
                "mode": CONFIG["mode"],
                "config_fingerprint": FINGERPRINT,
                "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            }, fh, indent=2)
    OUTPUT_DIR = RUN_DIR / "artifacts"
    CKPT_PATH = RUN_DIR / "checkpoints" / "ckpt.pt"
else:
    RUN_ID = None
    RUN_DIR = None
    OUTPUT_DIR = pathlib.Path(CONFIG["paths"]["output_dir"])
    CKPT_PATH = pathlib.Path(CONFIG["paths"]["checkpoint"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
CONES_DIR = OUTPUT_DIR / "cones"
CONES_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN_DIR = {RUN_DIR}\nOUTPUT_DIR = {OUTPUT_DIR}\nCKPT_PATH = {CKPT_PATH}\nRESUMED = {RESUMED}")

In [ ]:
# 5. Lightweight validation suite (CPU, mocks, no network) — must pass
# before any real-model work, exactly as in the earlier notebooks.
import subprocess

proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--no-header"],
    capture_output=True, text=True,
)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise RuntimeError("local test suite failed; fix before running the model")

In [ ]:
# 6. Verify the pilot reference (READ-ONLY): the recorded prompt hashes we
# must reproduce, the fitting recipe we must match, and the frozen pilot
# J_21 we will compare against. Nothing under the pilot run is written.
from jlens.lens import JacobianLens
from jlens.metadata import file_sha256

REF = CONFIG["reference"]
PILOT_DIR = RUNS_ROOT / REF["pilot_run_dir_name"]
PILOT_FIT_META_PATH = PILOT_DIR / REF["fit_metadata_relpath"]
if not PILOT_FIT_META_PATH.is_file():
    raise RuntimeError(
        f"pilot fit metadata not found at {PILOT_FIT_META_PATH}. In Colab, "
        "RUNS_ROOT must be the Drive runs/ directory containing the completed "
        f"pilot run {REF['pilot_run_dir_name']!r}."
    )
PILOT_FIT_META = json.load(open(PILOT_FIT_META_PATH, encoding="utf-8"))

PILOT_PROMPT_HASHES = PILOT_FIT_META["prompt_hashes"]
if len(PILOT_PROMPT_HASHES) != CONFIG["fitting"]["n_prompts"]:
    raise RuntimeError(
        f"pilot recorded {len(PILOT_PROMPT_HASHES)} prompt hashes, config "
        f"expects {CONFIG['fitting']['n_prompts']}"
    )

# The refit recipe must equal the pilot's, except source_layers = [21].
pilot_config = PILOT_FIT_META["config"]
for key in ("prompt_source", "n_prompts", "max_seq_len", "dim_batch", "seed",
            "checkpoint_every"):
    if pilot_config["fitting"][key] != CONFIG["fitting"][key]:
        raise RuntimeError(
            f"fitting.{key} mismatch: pilot={pilot_config['fitting'][key]!r} "
            f"vs diagnostic={CONFIG['fitting'][key]!r}"
        )
if pilot_config["positions"]["skip_first"] != CONFIG["positions"]["skip_first"]:
    raise RuntimeError("positions.skip_first mismatch vs pilot")
if pilot_config["sites"]["target_layer"] != CONFIG["sites"]["target_layer"]:
    raise RuntimeError("sites.target_layer mismatch vs pilot")
if 21 not in pilot_config["sites"]["source_layers"]:
    raise RuntimeError("pilot did not fit layer 21?!")
PILOT_REVISION = PILOT_FIT_META["load_info"]["model_revision"]
if PILOT_REVISION != REF["expect_model_revision"]:
    raise RuntimeError(
        f"pilot revision {PILOT_REVISION} != expected {REF['expect_model_revision']}"
    )

PILOT_LENS_PATH = PILOT_DIR / REF["lens_relpath"]
pilot_lens_sha = file_sha256(str(PILOT_LENS_PATH))
if REF["expect_lens_sha256"] and pilot_lens_sha != REF["expect_lens_sha256"]:
    raise RuntimeError(
        f"pilot lens fingerprint mismatch: expected {REF['expect_lens_sha256']}, "
        f"got {pilot_lens_sha}"
    )
PILOT_LENS = JacobianLens.load(str(PILOT_LENS_PATH))
PILOT_J21 = PILOT_LENS.jacobians[21].float()
if not torch.isfinite(PILOT_J21).all():
    raise RuntimeError("pilot J_21 contains non-finite values?!")
PILOT_J21_FRO = float(PILOT_J21.norm())

REFERENCE_VERIFICATION = {
    "pilot_run_dir": str(PILOT_DIR),
    "pilot_lens_sha256": pilot_lens_sha,
    "pilot_model_revision": PILOT_REVISION,
    "pilot_config_fingerprint": PILOT_FIT_META["config_fingerprint"],
    "n_prompt_hashes": len(PILOT_PROMPT_HASHES),
    "pilot_j21_frobenius_norm": PILOT_J21_FRO,
}
print(json.dumps(REFERENCE_VERIFICATION, indent=2))
print(f"pilot ||J_21||_F = {PILOT_J21_FRO:.3f} (the outlier under investigation)")

In [ ]:
# 7. PREFLIGHT — read this block before letting the fit start.
gpu_name, gpu_vram_gb = None, None
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name, gpu_vram_gb = props.name, round(props.total_memory / 2**30, 1)

print("=" * 68)
print("PREFLIGHT — layer-21 refit diagnostic")
print("=" * 68)
print(f"GPU:                {gpu_name or 'none visible'}"
      f"{f'  ({gpu_vram_gb} GB VRAM)' if gpu_vram_gb else ''}")
print(f"branch/commit:      {ENV.get('local_commit')}")
print(f"model:              {CONFIG['model']['repo_id']}")
print(f"revision (pinned):  {CONFIG['model']['revision']}")
print(f"source layers:      {CONFIG['sites']['source_layers']} (diagnostic scope)")
print(f"target layer:       {CONFIG['sites']['target_layer']}")
print(f"prompts:            {CONFIG['fitting']['n_prompts']} WikiText "
      f"(pilot hash-pinned), seed {CONFIG['fitting']['seed']}, "
      f"max_seq_len {CONFIG['fitting']['max_seq_len']}, "
      f"dim_batch {CONFIG['fitting']['dim_batch']}, "
      f"checkpoint every {CONFIG['fitting']['checkpoint_every']}")
print(f"pursuit:            layer 21, k={CONFIG['pursuit']['k_values']} only")
print(f"run directory:      {RUN_DIR or OUTPUT_DIR}")
print(f"fresh or resumed:   {'RESUMED (checkpoint continues)' if RESUMED else 'FRESH run'}")
print(f"model load allowed: {ALLOW_MODEL_LOAD}")
print("=" * 68)
if IN_COLAB and gpu_name is None:
    raise RuntimeError("no GPU visible in Colab — select an L4 runtime first")

In [ ]:
# 8. Load the SAME immutable Gemma revision the pilot used (gated).
# STOP CONDITIONS: wrong resolved revision; architecture mismatch; model
# parameters not frozen (verify_architecture raises on trainable params and
# records params_frozen=True only after checking every parameter).
MODEL = None
LOAD_INFO = None
ARCH_REPORT = None
if not ALLOW_MODEL_LOAD:
    print("JLENS_ALLOW_GEMMA != 1 — model loading disabled; light path only.")
else:
    from jlens.gemma4 import load_gemma4, resolve_revision, verify_architecture

    pinned = CONFIG["model"]["revision"]
    resolved = resolve_revision(CONFIG["model"]["repo_id"], pinned)
    if resolved != PILOT_REVISION:
        raise RuntimeError(
            f"resolved revision {resolved} != pilot revision {PILOT_REVISION}; "
            "refusing to fit a diagnostic on a different model"
        )
    print(f"{CONFIG['model']['repo_id']} pinned to {resolved}")

    _DTYPES = {"bfloat16": torch.bfloat16, "float32": torch.float32}
    MODEL, LOAD_INFO = load_gemma4(
        CONFIG["model"]["repo_id"],
        revision=resolved,
        dtype=_DTYPES[CONFIG["model"]["dtype"]],
        device_map=DEVICE_MAP,
        allow_model_load=True,
    )
    report = verify_architecture(
        MODEL,
        expect_n_layers=CONFIG["model"]["expect_n_layers"],
        expect_d_model=CONFIG["model"]["expect_d_model"],
        expect_vocab_size=CONFIG["model"]["expect_vocab_size"],
    )
    ARCH_REPORT = report.to_dict()
    if not ARCH_REPORT["params_frozen"]:
        raise RuntimeError("model parameters are not frozen; aborting")
    print(f"architecture verified: {ARCH_REPORT['model_class']} "
          f"({ARCH_REPORT['n_layers']}L, d={ARCH_REPORT['d_model']}, "
          f"V={ARCH_REPORT['vocab_size']}), params_frozen=True")

In [ ]:
# 9. Fitting corpus: the SAME 100 WikiText prompts in the SAME order.
# STOP CONDITION: the full ordered hash list must equal the pilot's
# recorded prompt_hashes — any drift in the streamed dataset aborts the run.
PROMPTS = None
if MODEL is None:
    print("model not loaded — skipping corpus load (light path).")
else:
    from jlens.examples import load_wikitext_prompts
    from jlens.metadata import prompt_hashes

    torch.manual_seed(CONFIG["fitting"]["seed"])  # mirrors scripts/fit_gemma.py
    PROMPTS = load_wikitext_prompts(n_prompts=CONFIG["fitting"]["n_prompts"])
    if len(PROMPTS) != CONFIG["fitting"]["n_prompts"]:
        raise RuntimeError(f"expected {CONFIG['fitting']['n_prompts']} prompts, "
                           f"got {len(PROMPTS)}")
    hashes = prompt_hashes(PROMPTS)
    if hashes != PILOT_PROMPT_HASHES:
        first_bad = next(i for i, (a, b) in enumerate(zip(hashes, PILOT_PROMPT_HASHES))
                         if a != b)
        raise RuntimeError(
            "fitting corpus does not reproduce the pilot corpus: first "
            f"mismatch at prompt {first_bad} "
            f"({hashes[first_bad]} != {PILOT_PROMPT_HASHES[first_bad]}). "
            "The streamed WikiText order has drifted; do not proceed."
        )
    print(f"{len(PROMPTS)} prompts loaded; all {len(hashes)} hashes match the "
          "pilot run in order.")

In [ ]:
# 10. Fit J_21 with per-prompt diagnostics (checkpointed + resume-safe).
# STOP CONDITION: a non-finite probe Jacobian aborts before the long fit.
NEW_LENS = None
RECORDER = None
FIT_RUNTIME = None
if MODEL is None:
    print("model not loaded — skipping fit (light path).")
else:
    from jlens.fit_diagnostics import PromptDiagnosticsRecorder
    from jlens.fitting import fit
    from jlens.gemma4 import probe_fit_cost

    probe = probe_fit_cost(
        MODEL, PROMPTS[0], CONFIG["sites"]["source_layers"],
        dim_batch=CONFIG["fitting"]["dim_batch"],
        max_seq_len=min(48, CONFIG["fitting"]["max_seq_len"]),
    )
    print(f"probe: {probe}")
    if not probe["all_finite"]:
        raise RuntimeError("probe produced non-finite Jacobians; aborting")

    DIAG_JSONL = OUTPUT_DIR / "per_prompt_diagnostics.jsonl"
    RECORDER = PromptDiagnosticsRecorder(
        str(DIAG_JSONL), expected_prompt_hashes=PILOT_PROMPT_HASHES,
    )
    if RECORDER.rows:
        print(f"resuming diagnostics: {len(RECORDER.rows)} rows already recorded")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    NEW_LENS = fit(
        MODEL, PROMPTS,
        source_layers=CONFIG["sites"]["source_layers"],
        target_layer=CONFIG["sites"]["target_layer"],
        dim_batch=CONFIG["fitting"]["dim_batch"],
        max_seq_len=CONFIG["fitting"]["max_seq_len"],
        skip_first=CONFIG["positions"]["skip_first"],
        checkpoint_path=str(CKPT_PATH),
        checkpoint_every=CONFIG["fitting"]["checkpoint_every"],
        resume=True,
        on_prompt=RECORDER.on_prompt,
    )
    FIT_RUNTIME = time.perf_counter() - t0
    print(f"fit complete: {NEW_LENS} in {FIT_RUNTIME:.0f}s "
          f"({len(RECORDER.rows)} diagnostic rows)")

In [ ]:
# 11. Post-fit: finite validation (bounded memory), save the new lens,
# write per-prompt + running-accumulation diagnostics, and record resolved
# execution provenance (configured vs resolved allow_model_load).
# STOP CONDITION: any non-finite value in the fitted J_21 aborts.
NEW_LENS_SHA = None
if NEW_LENS is None:
    print("no fitted lens — skipping (light path).")
else:
    from jlens.metadata import execution_record, prompt_hashes, write_metadata
    from jlens.pursuit import validate_finite

    NEW_J21 = NEW_LENS.jacobians[21].float()
    if not validate_finite(NEW_J21):
        raise RuntimeError("fitted J_21 contains NaN/Inf; aborting")

    LENS_PATH = OUTPUT_DIR / "lens.pt"
    NEW_LENS.save(str(LENS_PATH))
    NEW_LENS_SHA = file_sha256(str(LENS_PATH))
    reloaded = JacobianLens.load(str(LENS_PATH))
    assert reloaded.source_layers == [21] and reloaded.n_prompts == NEW_LENS.n_prompts

    RECORDER.write_csv(str(OUTPUT_DIR / "per_prompt_diagnostics.csv"))
    RECORDER.write_summary(str(OUTPUT_DIR / "running_accumulation.json"))
    ACCUMULATION = RECORDER.summary()

    EXECUTION = execution_record(
        configured_allow_model_load=CONFIG["model"]["allow_model_load"],
        resolved_allow_model_load=ALLOW_MODEL_LOAD,
        model_loaded=MODEL is not None,
        override_source="notebook:JLENS_ALLOW_GEMMA",
    )
    write_metadata(str(OUTPUT_DIR / "fit_metadata.json"), {
        "config_path": CONFIG_PATH,
        "config": CONFIG,
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
        "reference_verification": REFERENCE_VERIFICATION,
        "load_info": LOAD_INFO,
        "architecture_report": ARCH_REPORT,
        "probe": probe,
        "prompt_hashes": prompt_hashes(PROMPTS),
        "n_prompts_fitted": NEW_LENS.n_prompts,
        "fit_runtime_seconds": round(FIT_RUNTIME, 1),
        "peak_cuda_memory_gb": (
            round(torch.cuda.max_memory_allocated() / 2**30, 2)
            if torch.cuda.is_available() else None
        ),
        "lens_path": str(LENS_PATH),
        "lens_sha256": NEW_LENS_SHA,
        "environment": ENV,
    })
    print(f"lens saved: {LENS_PATH} ({NEW_LENS_SHA})")
    print(json.dumps(ACCUMULATION, indent=2)[:2500])

In [ ]:
# 12. Compare the refit J_21 with the frozen pilot J_21 (read-only).
# The three-way question this cell informs (descriptive, no verdict field):
# reproduced norm + stable accumulation -> property of the layer;
# divergent norm / non-stabilizing mean -> unstable corpus-averaged fit;
# top-share ~ total -> dominance by few prompts.
if NEW_LENS is None:
    print("no fitted lens — skipping comparison (light path).")
else:
    new_j = NEW_LENS.jacobians[21].float().cpu()
    old_j = PILOT_J21.cpu()
    dot = float((new_j * old_j).sum())
    cosine = dot / (float(new_j.norm()) * float(old_j.norm()))
    new_svals = torch.linalg.svdvals(new_j)
    old_svals = torch.linalg.svdvals(old_j)
    REFIT_COMPARISON = {
        "pilot_j21_frobenius_norm": float(old_j.norm()),
        "refit_j21_frobenius_norm": float(new_j.norm()),
        "frobenius_norm_ratio_refit_over_pilot": float(new_j.norm() / old_j.norm()),
        "cosine_similarity_flattened": cosine,
        "pilot_spectral_norm": float(old_svals[0]),
        "refit_spectral_norm": float(new_svals[0]),
        "relative_difference_frobenius": float((new_j - old_j).norm() / old_j.norm()),
        "refit_max_abs": float(new_j.abs().max()),
        "pilot_max_abs": float(old_j.abs().max()),
        "note": (
            "Pilot values are read from the frozen pilot lens; the pilot "
            "artifacts are never modified. Interpretation belongs in the "
            "follow-up analysis, not in this record."
        ),
    }
    with open(OUTPUT_DIR / "refit_comparison.json", "w", encoding="utf-8") as fh:
        json.dump(REFIT_COMPARISON, fh, indent=2)
    print(json.dumps(REFIT_COMPARISON, indent=2))

In [ ]:
# 13. Held-out evaluation at layer 21 ONLY (v2 prompt set, named controls).
# The layer-mapping controls need >= 2 fitted layers and are correctly
# omitted for this single-layer lens (jlens/logit_lens/permuted/random run).
EVAL_ROWS = None
if NEW_LENS is None:
    print("no fitted lens — skipping evaluation (light path).")
else:
    from jlens.evaluation import build_control_suite, evaluate_suite, load_eval_prompts_v2
    from jlens.gemma4 import softcap_disabled
    from jlens.metadata import write_metadata

    EVAL_ROWS = load_eval_prompts_v2(CONFIG["eval"]["prompts_path"], MODEL.tokenizer)
    suite = build_control_suite(NEW_LENS, control_seed=CONFIG["eval"]["control_seed"])
    print("variants:", ", ".join(suite))
    t0 = time.perf_counter()
    with softcap_disabled(MODEL):  # paper convention; rankings unaffected
        EVAL_RESULTS = evaluate_suite(
            MODEL, suite, EVAL_ROWS,
            layers=[21],
            top_k=CONFIG["eval"]["top_k"],
            max_seq_len=CONFIG["eval"]["max_seq_len"],
        )
    print(f"evaluated {EVAL_RESULTS['n_prompts']} prompts in "
          f"{time.perf_counter() - t0:.0f}s")
    write_metadata(str(OUTPUT_DIR / "eval_v2_results.json"), {
        "config_fingerprint": FINGERPRINT,
        "lens_sha256": NEW_LENS_SHA,
        "reference_verification": REFERENCE_VERIFICATION,
        "load_info": LOAD_INFO,
        "results": EVAL_RESULTS,
        "environment": ENV,
        "notes": (
            "Single-layer diagnostic lens: adjacent/distant/shuffled layer "
            "controls are omitted by build_control_suite (>= 2 layers needed)."
        ),
    })
    for fmt in ("plain", "chat"):
        agg = EVAL_RESULTS["aggregates"][fmt]["21"]
        line = "  ".join(
            f"{name}: mr={agg[name]['median_rank']:.0f} "
            f"h@10={agg[name]['hit_rate@10']:.2f}"
            for name in ("jlens", "logit_lens", "permuted", "random")
            if name in agg
        )
        print(f"L21 {fmt}  {line}")

In [ ]:
# 14. J-space gradient pursuit on the refit J_21 — layer 21, k=10 ONLY —
# with the lower-memory helpers (chunked dictionary build; chunked finite
# checks happen inside JSpaceDictionary). Cone records use the same schema
# as the completed jspace run so scripts/analyze_jspace.py tooling applies.
CAPTURE_META = None
if NEW_LENS is None:
    print("no fitted lens — skipping pursuit (light path).")
else:
    from jlens.cones import make_cone_record, save_cone_records
    from jlens.evaluation import capture_residuals
    from jlens.metadata import prompt_hashes as _hashes
    from jlens.pursuit import JSpaceDictionary, PursuitSettings, gradient_pursuit

    PUR = CONFIG["pursuit"]
    CAPTURE_META = []
    chunks = []
    t0 = time.perf_counter()
    for row_idx, row in enumerate(EVAL_ROWS):
        residuals, model_logits, input_ids = capture_residuals(
            MODEL, row["text"], layers=[21],
            positions=row["positions"],
            max_seq_len=CONFIG["eval"]["max_seq_len"],
        )
        row_hash = _hashes([row["text"]])[0]
        top1 = model_logits.argmax(-1)
        seq_len = input_ids.shape[1]
        for i, pos in enumerate(row["positions"]):
            tok_id = int(input_ids[0, pos])
            CAPTURE_META.append({
                "slug": row["slug"], "category": row["category"],
                "format": row["format"], "position": int(pos),
                "prompt_hash": row_hash, "seq_len": int(seq_len),
                "input_token_id": tok_id,
                "input_token": MODEL.tokenizer.decode([tok_id]),
                "model_top1_id": int(top1[i]),
                "model_top1_token": MODEL.tokenizer.decode([int(top1[i])]),
            })
        chunks.append(residuals[21].cpu())
        print(f"  [{row_idx + 1}/{len(EVAL_ROWS)}] {row['slug']} ({row['format']}) "
              f"seq_len={seq_len}  elapsed={time.perf_counter() - t0:.0f}s",
              flush=True)
    RESIDUALS_21 = torch.cat(chunks, dim=0)
    assert torch.isfinite(RESIDUALS_21).all(), "non-finite h at L21"
    print(f"captured {RESIDUALS_21.shape[0]} activations")

    RUN_PROVENANCE = {
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
        "config_fingerprint": FINGERPRINT,
        "lens_fingerprint": NEW_LENS_SHA,
        "lens_path": str(OUTPUT_DIR / "lens.pt"),
        "lens_n_prompts": NEW_LENS.n_prompts,
        "model_revision": LOAD_INFO["model_revision"],
        "local_commit": ENV.get("local_commit"),
        "upstream_commit": ENV.get("upstream_commit"),
        "diagnostic": "layer21 refit lens — NOT the frozen pilot lens",
    }
    W_U = MODEL._lm_head.weight
    t_dict = time.perf_counter()
    dictionary = JSpaceDictionary.from_lens(
        NEW_LENS, 21, W_U,
        device=W_U.device,
        dtype={"float32": torch.float32, "float16": torch.float16}[PUR["atoms_dtype"]],
        build_chunk_rows=PUR["build_chunk_rows"],  # lower-memory build (L4)
    )
    print(f"dictionary {dictionary.n_atoms}x{dictionary.d_model} built in "
          f"{time.perf_counter() - t_dict:.1f}s", flush=True)

    k = PUR["k_values"][0]
    settings = PursuitSettings(
        k=k,
        normalize_atoms=PUR["normalize_atoms"],
        refine_steps=PUR["refine_steps"],
        tol_relative_residual=float(PUR["tol_relative_residual"]),
        correlation_chunk_size=PUR["correlation_chunk_size"],
    )
    t_unit = time.perf_counter()
    result = gradient_pursuit(RESIDUALS_21, dictionary, settings)
    cone_records = []
    for meta, record in zip(CAPTURE_META, result.to_records(), strict=True):
        labels = [MODEL.tokenizer.decode([i]) for i in record["token_ids"]]
        cone_records.append(make_cone_record(
            record,
            decoded_labels=labels,
            layer=21,
            position=meta["position"],
            input_token_id=meta["input_token_id"],
            input_token=meta["input_token"],
            prompt_hash=meta["prompt_hash"],
            prompt_slug=meta["slug"],
            prompt_format=meta["format"],
            run_provenance={**RUN_PROVENANCE,
                            "category": meta["category"],
                            "model_top1_id": meta["model_top1_id"],
                            "model_top1_token": meta["model_top1_token"]},
        ))
    out_path = CONES_DIR / f"cones_layer21_k{k}.json"
    tmp_path = out_path.with_suffix(".json.tmp")
    save_cone_records(cone_records, str(tmp_path))
    os.replace(tmp_path, out_path)
    mean_expl = sum(r["reconstruction"]["explained_fraction"]
                    for r in cone_records) / len(cone_records)
    print(f"L21 k={k}: {len(cone_records)} activations in "
          f"{time.perf_counter() - t_unit:.1f}s  mean_explained={mean_expl:.6f}  "
          f"(frozen pilot lens gave 2.3e-05)  -> {out_path}", flush=True)
    del dictionary
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# 15. Run manifest + human-readable summary. Writing run_metadata.json marks
# the run COMPLETE (resume attempts will then refuse this directory).
if NEW_LENS is None:
    print("nothing to summarize (light path).")
else:
    from jlens.metadata import write_metadata

    manifest = {
        "run_id": RUN_ID,
        "run_dir": str(RUN_DIR),
        "mode": "layer21_diagnostic",
        "config": CONFIG,
        "config_fingerprint": FINGERPRINT,
        "execution": EXECUTION,
        "reference_verification": REFERENCE_VERIFICATION,
        "refit_comparison": REFIT_COMPARISON,
        "accumulation_summary": ACCUMULATION,
        "load_info": LOAD_INFO,
        "architecture_report": ARCH_REPORT,
        "lens_sha256": NEW_LENS_SHA,
        "n_activations_decomposed": len(CAPTURE_META),
        "capture_meta": CAPTURE_META,
        "environment": ENV,
        "notes": (
            "Layer-21 refit diagnostic. The pilot/jspace/smoke runs are "
            "read-only references and were not modified. Per-prompt "
            "diagnostics are descriptive statistics of the corpus-averaged "
            "estimator, not model-internal claims."
        ),
    }
    write_metadata(str((RUN_DIR or OUTPUT_DIR) / "run_metadata.json"), manifest)

    lines = [
        f"# Run {RUN_ID or OUTPUT_DIR}",
        "",
        "- mode: layer21_diagnostic (exact pilot-recipe refit of J_21 with "
        "per-prompt diagnostics)",
        f"- model: {LOAD_INFO['model_repo_id']} @ {LOAD_INFO['model_revision']}",
        f"- new lens: artifacts/lens.pt ({NEW_LENS_SHA}), layer 21 only, "
        f"{NEW_LENS.n_prompts} prompts",
        f"- pilot ||J_21||_F = {REFIT_COMPARISON['pilot_j21_frobenius_norm']:.3f}; "
        f"refit ||J_21||_F = {REFIT_COMPARISON['refit_j21_frobenius_norm']:.3f}; "
        f"cosine = {REFIT_COMPARISON['cosine_similarity_flattened']:.4f}",
        f"- top-1 prompt norm share = "
        f"{ACCUMULATION['top1_share_of_total_norm_mass']:.4f}; "
        f"running-mean tail change = "
        f"{ACCUMULATION['running_mean_frobenius_norm']['relative_change_over_tail_window']}",
        "- artifacts: lens.pt, per_prompt_diagnostics.{jsonl,csv}, "
        "running_accumulation.json, refit_comparison.json, "
        "eval_v2_results.json, cones/cones_layer21_k10.json",
        "",
        "Interpretation boundaries: docs/jspace_run_report.md §4 and "
        "docs/jspace_decomposition.md.",
    ]
    with open((RUN_DIR or OUTPUT_DIR) / "summary.md", "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))
    print("\n".join(lines))

## Expected wall time and follow-up

One prompt costs one forward + 320 backward passes (`d_model` 2560 /
`dim_batch` 8); the pilot averaged ~97 s/prompt on an L4 across seven
layers' gradient extraction — a single source layer is cheaper, but budget
1.5–3 h for the fit plus a few minutes for evaluation and the k=10 pursuit.
The fit checkpoints every 5 prompts and resumes safely
(`L21DIAG_RESUME_RUN_DIR`).

Afterwards, analyze the run offline (no GPU) with the existing tooling:
`per_prompt_diagnostics.csv` and `running_accumulation.json` answer the
stability/dominance question; `refit_comparison.json` answers
reproducibility; the cone records can be compared against the frozen-lens
run with `jlens/similarity.py`. Do **not** promote either lens: the frozen
pilot lens remains the authoritative artifact until a decision is made in
the research log.